In [105]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import pandas as pd
import zipfile
import os
import time
import gc
from PIL import Image
from tqdm.notebook import tqdm
from utils import MetricsEngine, visual_compare  # Assumes your utils file is present
import torch.nn.functional as F

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Using device: cuda
Initial VRAM Allocated: 378.97 MB


In [106]:
def load_and_prep_data(image_path, downscale_factor=4):
    """Loads image, generates raw coords, and locks everything to GPU."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    new_w, new_h = w // downscale_factor, h // downscale_factor
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    img_np = np.array(img)
    img_norm = img_np / 255.0
    
    # 1. Create Raw Coordinates
    y_coords = np.linspace(-1, 1, new_h)
    x_coords = np.linspace(-1, 1, new_w)
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    coords_raw = torch.tensor(np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1), dtype=torch.float32)
    
    # 2. Colors
    colors = torch.tensor(img_norm.reshape(-1, 3), dtype=torch.float32)
    
    # 3. Push EVERYTHING to GPU permanently
    return {
        "coords_raw": coords_raw.to(device),
        "colors": colors.to(device),
        "h": new_h, "w": new_w,
        "num_pixels": new_h * new_w,
        "original_np": img_np
    }

# Load Data (Adjust downscale_factor based on your image)
dataset = load_and_prep_data("jcsmr-1.jpg", downscale_factor=4)
print(f"Resolution: {dataset['w']}x{dataset['h']} | Total Pixels: {dataset['num_pixels']}")
print(f"VRAM after data load: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

Resolution: 1404x936 | Total Pixels: 1314144
VRAM after data load: 378.04 MB


In [107]:
# --- ARCHITECTURE: PURE PYTORCH INSTANT-NGP (Multi-Res Grid) ---
# This gives you the insane speed of Hash Grids without C++ compilation errors!

class HashGrid2D(nn.Module):
    def __init__(
        self,
        num_levels=6,
        base_res=16,
        max_res=384,
        feature_dim=2,
        log2_hashmap_size=16  # BIG FIX
    ):
        super().__init__()
        self.num_levels = num_levels
        self.feature_dim = feature_dim
        self.hashmap_size = 2 ** log2_hashmap_size

        growth_factor = np.exp((np.log(max_res) - np.log(base_res)) / (num_levels - 1))
        self.resolutions = torch.tensor(
            [int(base_res * (growth_factor ** i)) for i in range(num_levels)],
            device=device
        )

        self.hash_tables = nn.ParameterList([
            nn.Parameter(torch.randn(self.hashmap_size, feature_dim) * 0.01)
            for _ in range(num_levels)
        ])

        self.decoder = nn.Sequential(
            nn.Linear(num_levels * feature_dim, 48),  # slightly bigger
            nn.ReLU(),
            nn.Linear(48, 48),
            nn.ReLU(),
            nn.Linear(48, 3)
        )

    def hash_fn(self, coords):
        primes = torch.tensor([1, 2654435761], device=coords.device)
        x = coords * primes
        x = torch.bitwise_xor(x[:, 0], x[:, 1])
        return x % self.hashmap_size

    def forward(self, coords):
        coords = (coords + 1) / 2  # [0,1]

        features = []

        for lvl in range(self.num_levels):
            res = self.resolutions[lvl]
            table = self.hash_tables[lvl]

            pos = coords * res
            pos_floor = torch.floor(pos).int()

            x0 = pos_floor[:, 0]
            y0 = pos_floor[:, 1]
            x1 = x0 + 1
            y1 = y0 + 1

            wx = pos[:, 0] - x0.float()
            wy = pos[:, 1] - y0.float()

            h00 = self.hash_fn(torch.stack([x0, y0], dim=-1))
            h10 = self.hash_fn(torch.stack([x1, y0], dim=-1))
            h01 = self.hash_fn(torch.stack([x0, y1], dim=-1))
            h11 = self.hash_fn(torch.stack([x1, y1], dim=-1))

            f00 = table[h00]
            f10 = table[h10]
            f01 = table[h01]
            f11 = table[h11]

            f0 = f00 * (1 - wx).unsqueeze(-1) + f10 * wx.unsqueeze(-1)
            f1 = f01 * (1 - wx).unsqueeze(-1) + f11 * wx.unsqueeze(-1)
            f = f0 * (1 - wy).unsqueeze(-1) + f1 * wy.unsqueeze(-1)

            features.append(f)

        x = torch.cat(features, dim=-1)
        x = self.decoder(x)

        return torch.clamp(x, 0, 1)

In [108]:
def train_and_evaluate(config, dataset):
    model_name = config["name"]
    print(f"\n=== Starting: {model_name} ===")
    
    # Instant-NGP Multi-Res Grid
    model = HashGrid2D(
        num_levels=config["levels"],
        base_res=16,
        max_res=384,              # important: don't go too high
        feature_dim=config["feat_dim"],
        log2_hashmap_size=16      # try 13–15 range
    ).to(device)
    
    input_data = dataset["coords_raw"]
        
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    # Cosine Annealing Scheduler (Decays LR smoothly to 1e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-6)
    criterion = nn.MSELoss()
    
    # Automatic Mixed Precision Scaler
    scaler = torch.amp.GradScaler('cuda')
    
    batch_size = config["batch_size"]
    num_pixels = dataset["num_pixels"]
    
    model.train()
    pbar = tqdm(range(config["epochs"]), desc=model_name)
    
    for epoch in pbar:
        # Pre-shuffled Array (Done instantly on GPU)
        indices = torch.randperm(num_pixels, device=device)
        
        epoch_loss = 0.0
        batches = 0
        
        # Sequential Slicing (Zero GPU memory reallocation)
        for i in range(0, num_pixels, batch_size):
            batch_idx = indices[i : i + batch_size]
            b_coords = input_data[batch_idx]
            b_colors = dataset["colors"][batch_idx]
            
            optimizer.zero_grad(set_to_none=True) # Slightly faster than standard zero_grad
            
            # AMP Forward Pass
            with torch.amp.autocast('cuda'):
                pred = model(b_coords)
                loss = criterion(pred, b_colors)
                
            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.detach()
            batches += 1
            
        scheduler.step()
        
        # Update UI every 50 epochs (Avoids stalling the pipeline)
        if epoch % 50 == 0:
            avg_loss = epoch_loss / batches
            psnr = -10.0 * torch.log10(avg_loss).item()
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({"PSNR": f"{psnr:.2f}", "LR": f"{current_lr:.1e}"})

    # --- EVALUATION STAGE ---
    model.eval()
    start_time = time.time()
    predicted_colors = []
    
    # Chunked Inference with AMP
    with torch.no_grad(), torch.amp.autocast('cuda'):
        chunk_size = 65536 
        for i in range(0, num_pixels, chunk_size):
            chunk = input_data[i : i + chunk_size]
            predicted_colors.append(model(chunk))
            
        full_pred = torch.cat(predicted_colors, dim=0)
        
    torch.cuda.synchronize()
    latency_ms = (time.time() - start_time) * 1000

    # Save Output Image
    pred_np = full_pred.float().cpu().numpy().reshape(dataset["h"], dataset["w"], 3)
    pred_img_uint8 = (pred_np * 255).clip(0, 255).astype(np.uint8)
    os.makedirs("images", exist_ok=True)
    cv2.imwrite(f"images/{model_name}.png", cv2.cvtColor(pred_img_uint8, cv2.COLOR_RGB2BGR))

    # --- EXTREME STORAGE MINIMIZATION ---
    os.makedirs("models", exist_ok=True)
    pth_path = f"models/{model_name}.pth"
    zip_path = f"models/{model_name}.zip"
    
    # Convert weights to FP16 before saving (Halves the size immediately)
    model.half()
    torch.save(model.state_dict(), pth_path)
    
    # Compress using LZMA (Aggressive compression for numbers)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_LZMA) as zipf:
        zipf.write(pth_path, arcname=f"{model_name}.pth")
    size_kb = os.path.getsize(zip_path) / 1024.0

    # Compute Metrics
    engine = MetricsEngine(device)
    metrics = engine.compute_all(dataset["original_np"], pred_img_uint8)
    
    row = {"Method": model_name, "Size_KB": round(size_kb, 2), "Latency_ms": round(latency_ms, 2)}
    row.update(metrics)
    
    # --- CLEAN SLATE PROTOCOL (VRAM PURGE) ---
    del model
    del optimizer
    del scheduler
    del scaler
    del full_pred
    del predicted_colors
    del criterion
    gc.collect()
    torch.cuda.empty_cache()
    
    return row

In [109]:
# Batch size can be very large now because we use Mixed Precision!
#torch.cuda.empty_cache()
#batchsize = dataset["num_pixels"] # Process the whole image at once!
batchsize = 65536

# Instant-NGP Variations (Train in seconds, MASSIVE PSNR)
EXPERIMENTS = [
    {
        "name": "HashNGP_Tiny",
        "levels": 6,
        "feat_dim": 2,
        "epochs": 900,
        "batch_size": batchsize,
        "lr": 1e-2
    }
]

os.makedirs("results", exist_ok=True) 
all_neural_results = []

for config in EXPERIMENTS: 
    result = train_and_evaluate(config, dataset)
    all_neural_results.append(result)
    
    # Save intermediate in case of crash
    pd.DataFrame(all_neural_results).to_csv("results/neural_metrics_temp.csv", index=False)
    print(f"Finished {config['name']} | VRAM Reset to: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

# Final Save
df_neural = pd.DataFrame(all_neural_results)
df_neural.to_csv("results/neural_metrics.csv", index=False) 
print("\nAll experiments completed and fully optimized!") 
display(df_neural)


=== Starting: HashNGP_Tiny ===


HashNGP_Tiny:   0%|          | 0/900 [00:00<?, ?it/s]

Finished HashNGP_Tiny | VRAM Reset to: 378.04 MB

All experiments completed and fully optimized!


,Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
0,HashNGP_Tiny,1411.82,163.67,33.98838,0.923784,0.996255,0.253019
